# Somalia Economic Dashboard
## Integrated Summary for Market Intelligence

This notebook combines exchange rate, inflation, telecom, and cost-of-living analytics into a cohesive dashboard-style summary.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries loaded")

## 1. Load All Processed Data

In [ ]:
data_dir = '../data/processed'
exchange_rates = pd.read_csv(f'{data_dir}/exchange_rates.csv', parse_dates=['date'])
fuel_prices = pd.read_csv(f'{data_dir}/fuel_prices.csv', parse_dates=['date'])
food_prices = pd.read_csv(f'{data_dir}/food_prices.csv', parse_dates=['date'])
telecom_packages = pd.read_csv(f'{data_dir}/telecom_packages.csv', parse_dates=['date'])
cost_of_living = pd.read_csv(f'{data_dir}/cost_of_living.csv', parse_dates=['date'])

print('Datasets loaded successfully')
print(f'Exchange rates: {len(exchange_rates):,}')
print(f'Fuel prices: {len(fuel_prices):,}')
print(f'Food prices: {len(food_prices):,}')
print(f'Telecom packages: {len(telecom_packages):,}')
print(f'Cost of living: {len(cost_of_living):,}')

## 2. KPI Summary

In [ ]:
latest_date = exchange_rates['date'].max()
latest_rate = exchange_rates.loc[exchange_rates['date'] == latest_date, 'usd_sos_rate'].mean()
avg_inflation = food_prices['inflation_percent'].mean()
avg_telecom = telecom_packages.loc[telecom_packages['date'] == telecom_packages['date'].max(), 'package_price_usd'].mean()
latest_col = cost_of_living.loc[cost_of_living['date'] == cost_of_living['date'].max(), 'total_cost_index'].mean()

kpis = pd.DataFrame({
    'Metric': ['USD/SOS Rate', 'Average Food Inflation', 'Average Telecom Price', 'Average Total Cost of Living'],
    'Value': [round(latest_rate, 2), f'{round(avg_inflation, 2)}%', f'${round(avg_telecom, 2)}', round(latest_col, 2)]
})
kpis

## 3. Exchange Rate Trend

In [ ]:
exchange_month = exchange_rates.set_index('date').resample('M')['usd_sos_rate'].mean().reset_index()
fig = px.line(exchange_month, x='date', y='usd_sos_rate', title='Monthly Average USD/SOS Exchange Rate')
fig.write_html('../visuals/dashboard_exchange_rate_trend.html')
fig.show()

## 4. Food Inflation Snapshot

In [ ]:
top_inflation = food_prices.groupby('product_name')['inflation_percent'].mean().sort_values(ascending=False).head(8).reset_index()
fig = px.bar(top_inflation, x='inflation_percent', y='product_name', orientation='h', title='Top Food Inflation Products')
fig.write_html('../visuals/dashboard_food_inflation.html')
fig.show()

## 5. Telecom Price Landscape

In [ ]:
telecom_latest = telecom_packages[telecom_packages['date'] == telecom_packages['date'].max()]
provider_avg = telecom_latest.groupby('provider')['package_price_usd'].mean().sort_values().reset_index()
fig = px.bar(provider_avg, x='package_price_usd', y='provider', orientation='h', title='Average Telecom Price by Provider (Latest)')
fig.write_html('../visuals/dashboard_telecom_provider.html')
fig.show()

## 6. Cost of Living Comparison

In [ ]:
latest_col_cities = cost_of_living.loc[cost_of_living['date'] == cost_of_living['date'].max()].sort_values('total_cost_index', ascending=False)
fig = px.bar(latest_col_cities, x='total_cost_index', y='city', orientation='h', title='Latest Cost of Living Index by City')
fig.write_html('../visuals/dashboard_col_by_city.html')
fig.show()

## 7. Regional Comparison

In [ ]:
city_pricing = latest_col_cities[['city', 'total_cost_index']].merge(
    exchange_rates.loc[exchange_rates['date'] == latest_date, ['city', 'usd_sos_rate']],
    on='city',
    how='left'
)
fig = px.scatter(city_pricing, x='usd_sos_rate', y='total_cost_index', color='city', size='total_cost_index', title='Regional Exchange Rate vs Cost of Living')
fig.write_html('../visuals/dashboard_regional_scatter.html')
fig.show()

## 8. Conclusion

- Somalia faces ongoing currency depreciation and food inflation pressures.
- Telecom pricing is competitive but exhibits regional gaps.
- Cost-of-living differences are significant across major cities.
- This dashboard provides a foundation for operational monitoring and market insight.